<a href="https://colab.research.google.com/github/Clei21/Projeto2/blob/main/experimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Trade-off entre Especialização e Generalização em LLMs via Fine-Tuning


## 1. Ambiente

In [1]:
import glob
import json
import os

from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/tp2'
os.makedirs(DRIVE_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
%cd /content
!rm -rf Projeto2
!git clone https://github.com/Clei21/Projeto2.git
%cd Projeto2

!pip install -q -r requirements.txt

# Remove pacotes pré-instalados do Colab incompatíveis com as versões fixadas no projeto
!pip uninstall -y -q torchvision torchaudio tensorflow tf-keras jax jaxlib flax

/content
Cloning into 'Projeto2'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 98 (delta 19), reused 79 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 72.47 KiB | 18.12 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/Projeto2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 161.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 31.6 MB/s eta 0

> Caso o Colab solicite **reiniciar a sessão** após a instalação, reinicie e execute
> novamente as células da seção 1.

## 2. Dados

Obtém o Spider (mantido em `MyDrive/tp2/spider_data.zip`; baixado do site oficial apenas
na primeira execução), restaura artefatos de sessões anteriores e executa o
pré-processamento quando necessário.

In [3]:
if not os.path.exists(f'{DRIVE_DIR}/spider_data.zip'):
    !pip install -q gdown
    !gdown 1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J -O {DRIVE_DIR}/spider_data.zip

!unzip -q -o {DRIVE_DIR}/spider_data.zip
if os.path.exists('spider_data'):
    !mv spider_data spider

!cp -r {DRIVE_DIR}/data . 2>/dev/null || true
!cp -r {DRIVE_DIR}/results . 2>/dev/null || true
!cp -r {DRIVE_DIR}/outputs . 2>/dev/null || true

if not os.path.exists('data/dev.jsonl'):
    !python scripts/prepare_spider.py --spider_dir spider --out_dir data
    !python scripts/build_mmlu_suite.py --out data/mmlu_suite.json
    !cp -r data {DRIVE_DIR}/

!ls spider | head -5
!ls data

database
dev_gold.sql
dev.json
README.txt
tables.json
dev.jsonl  few_shot.json  mmlu_suite.json  train.jsonl


In [4]:
!ls outputs/lora_a 2>/dev/null || echo "TREINO A: não concluído"
!ls results/ 2>/dev/null

adapter_config.json	   checkpoint-874	    tokenizer_config.json
adapter_model.safetensors  merges.txt		    tokenizer.json
added_tokens.json	   README.md		    training_args.bin
checkpoint-437		   special_tokens_map.json  vocab.json
eval_predictions_baseline.json	mmlu_baseline.json  predictions_baseline.jsonl


## 3. Baseline no Spider — Fase 2 (~2–4 h)

Avalia o modelo base no dev split completo com o prompt few-shot fixo e decodificação
greedy; em seguida aplica a métrica de Execution Accuracy via pytest.

In [ ]:
!python scripts/generate_predictions.py \
  --model_name Qwen/Qwen2.5-3B-Instruct \
  --dev_file data/dev.jsonl --few_shot data/few_shot.json \
  --out results/predictions_baseline.jsonl

!PREDICTIONS=results/predictions_baseline.jsonl SPIDER_DB_DIR=spider/database pytest tests/test_spider_eval.py -s
!cp -r results {DRIVE_DIR}/

## 4. MMLU do modelo base — Fase 5 (~20 min)

In [ ]:
!python scripts/evaluate_mmlu.py --model_name Qwen/Qwen2.5-3B-Instruct \
  --suite data/mmlu_suite.json --out results/mmlu_baseline.json
!cp -r results {DRIVE_DIR}/

## 5. Fine-tuning — Fase 3 (~4–6 h por configuração)

Duas configurações de hiperparâmetros (configs/lora_a.yaml e configs/lora_b.yaml),
variando a taxa de aprendizado em uma ordem de grandeza. Recomenda-se executar uma
configuração por sessão, dado o limite de GPU do Colab gratuito.

In [ ]:
!python scripts/train_lora.py --config configs/lora_a.yaml
!cp -r outputs {DRIVE_DIR}/

In [ ]:
!python scripts/train_lora.py --config configs/lora_b.yaml
!cp -r outputs {DRIVE_DIR}/

## 6. Avaliação dos modelos fine-tuned no Spider — Fase 4 (~2–4 h por modelo)

Procedimento idêntico ao baseline (mesmo prompt, mesma métrica, mesma decodificação),
carregando o adaptador LoRA correspondente.

In [ ]:
!python scripts/generate_predictions.py \
  --model_name Qwen/Qwen2.5-3B-Instruct --adapter outputs/lora_a \
  --dev_file data/dev.jsonl --few_shot data/few_shot.json \
  --out results/predictions_lora_a.jsonl

!PREDICTIONS=results/predictions_lora_a.jsonl SPIDER_DB_DIR=spider/database pytest tests/test_spider_eval.py -s
!cp -r results {DRIVE_DIR}/

In [ ]:
!python scripts/generate_predictions.py \
  --model_name Qwen/Qwen2.5-3B-Instruct --adapter outputs/lora_b \
  --dev_file data/dev.jsonl --few_shot data/few_shot.json \
  --out results/predictions_lora_b.jsonl

!PREDICTIONS=results/predictions_lora_b.jsonl SPIDER_DB_DIR=spider/database pytest tests/test_spider_eval.py -s
!cp -r results {DRIVE_DIR}/

## 7. MMLU dos modelos fine-tuned — Fase 5 (~20 min por modelo)

In [ ]:
!python scripts/evaluate_mmlu.py --model_name Qwen/Qwen2.5-3B-Instruct \
  --adapter outputs/lora_a --suite data/mmlu_suite.json --out results/mmlu_lora_a.json

!python scripts/evaluate_mmlu.py --model_name Qwen/Qwen2.5-3B-Instruct \
  --adapter outputs/lora_b --suite data/mmlu_suite.json --out results/mmlu_lora_b.json

!cp -r results {DRIVE_DIR}/

## 8. Consolidação do trade-off — Fase 5

In [ ]:
!python scripts/analyze_tradeoff.py \
  --spider_baseline results/eval_predictions_baseline.json \
  --spider_finetuned results/eval_predictions_lora_a.json \
  --mmlu_baseline results/mmlu_baseline.json \
  --mmlu_finetuned results/mmlu_lora_a.json \
  --out results/tradeoff_lora_a.json

In [ ]:
!python scripts/analyze_tradeoff.py \
  --spider_baseline results/eval_predictions_baseline.json \
  --spider_finetuned results/eval_predictions_lora_b.json \
  --mmlu_baseline results/mmlu_baseline.json \
  --mmlu_finetuned results/mmlu_lora_b.json \
  --out results/tradeoff_lora_b.json

!cp -r results {DRIVE_DIR}/

## 9. Análise de erros

Lista falhas do modelo fine-tuned A no Spider dev (pergunta, SQL gerada e SQL de
referência), como insumo para a seção de análise de erros do relatório.

In [ ]:
with open('results/eval_predictions_lora_a.json', encoding='utf-8') as f:
    eval_data = json.load(f)

with open('results/predictions_lora_a.jsonl', encoding='utf-8') as f:
    preds = {(p['db_id'], p['question']): p for p in map(json.loads, f)}

erros = [d for d in eval_data['details'] if d['score'] == 0.0]
print(f"{len(erros)} falhas em {eval_data['n']} exemplos\n")

for d in erros[:5]:
    p = preds[(d['db_id'], d['question'])]
    print('=' * 80)
    print('Banco:    ', d['db_id'])
    print('Pergunta: ', d['question'])
    print('Motivo:   ', d['reason'])
    print('Gerada:   ', p['raw_output'][:300])
    print('Gold:     ', p['gold_sql'])

## 10. Encerramento

Resultados para o relatório (em `results/` e espelhados em `MyDrive/tp2/results/`):
Execution Accuracy em `eval_predictions_*.json`, acurácias do MMLU em `mmlu_*.json`
e a consolidação em `tradeoff_*.json`. Hardware utilizado: GPU NVIDIA T4 (16 GB VRAM),
Google Colab.